# 04d. Avaliar faixa `[0,05, 0,25)`

Diagnóstico do buraco `com_bloco_p_lt_025` do [`04c_decompor_sem_cpf.ipynb`](04c_decompor_sem_cpf.ipynb):
Censo sem CPF cujo melhor par no parquet está em `0,05 ≤ p < 0,25`.

Exige [`02b_aplicar_splink.ipynb`](02b_aplicar_splink.ipynb) com
`threshold_match_probability = 0,05` (senão a faixa vem vazia). Não muda a
escada do [`04_atribuir.ipynb`](04_atribuir.ipynb). Níveis de nome/data na
mesma ordem do treino (`02`).


In [1]:
import sys
from pathlib import Path

from IPython.display import display

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    SPLINK_ATRIBUICAO,
    SPLINK_INPUT_VIEW,
    SPLINK_PREDICTIONS,
    TABELA_CENSO_LIMPA_APLICACAO,
    TABELA_CPF_LIMPA_APLICACAO,
    drop_splink_temp_tables,
    get_connection,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)

PISO = 0.05
TETO = 0.25

print_paths()
require_input(SPLINK_PREDICTIONS, label='SPLINK_PREDICTIONS (02b com predict 0,05)')
require_input(SPLINK_ATRIBUICAO, label='SPLINK_ATRIBUICAO (rode o 04_atribuir antes)')
con = get_connection()
drop_splink_temp_tables(con)
require_tables(
    con,
    [TABELA_CENSO_LIMPA_APLICACAO, TABELA_CPF_LIMPA_APLICACAO],
    notebook_origem='00b',
)
materialize_splink_input(
    con,
    censo_table=TABELA_CENSO_LIMPA_APLICACAO,
    cpf_table=TABELA_CPF_LIMPA_APLICACAO,
)
print('Faixa:', PISO, '<= p <', TETO)


OUTPUT_DIR_BASE: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output
OUTPUT_DIR: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/uf_21
recorte: uf_21
CPF_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf.parquet
CENSO_PESSOAS_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_pessoas_2022.parquet
CENSO_PES_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_pes_nome.parquet
CPF_CPF_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf_cpf_nome.parquet
CENSO_ESPECIE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_especie.parquet
CENSO_ENDERECO_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_endereco.parquet
CENSO_FACE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_face.parquet
CENSO_LOGR_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_

splink_input → censo_limpo_aplicacao ∪ cpf_limpo_aplicacao
Faixa: 0.05 <= p < 0.25


In [2]:
con.execute(f'''
CREATE OR REPLACE VIEW splink_predictions AS
SELECT
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_l
        ELSE unique_id_r
    END AS unique_id_censo,
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_r
        ELSE unique_id_l
    END AS unique_id_cpf,
    match_probability,
    match_weight
FROM read_parquet('{SPLINK_PREDICTIONS}')
''')

con.execute(f'''
CREATE OR REPLACE VIEW atribuicao_2 AS
SELECT unique_id_censo, unique_id_cpf, match_probability, degrau, regra
FROM read_parquet('{SPLINK_ATRIBUICAO}')
''')

n_pred = con.execute('SELECT COUNT(*) FROM splink_predictions').fetchone()[0]
n_atrib = con.execute('SELECT COUNT(*) FROM atribuicao_2').fetchone()[0]
p_min = con.execute('SELECT MIN(match_probability) FROM splink_predictions').fetchone()[0]
print('Pares no parquet:', f'{n_pred:,}')
print('Atribuições:', f'{n_atrib:,}')
print('MIN(p) no parquet:', p_min)
if p_min is not None and p_min >= PISO + 1e-9:
    print(
        'AVISO: MIN(p) >=', PISO,
        '— rode o 02b com threshold_match_probability =', PISO,
    )


Pares no parquet: 6,598,211
Atribuições: 4,092,388
MIN(p) no parquet: 0.0500175142933243
AVISO: MIN(p) >= 0.05 — rode o 02b com threshold_match_probability = 0.05


Sem CPF no 04; melhor par na faixa `[0,05, 0,25)`. Conta também quem continua
sem par ≥ 0,05 (ainda abaixo do predict).


In [3]:
con.execute(f'''
CREATE OR REPLACE TABLE censo_sem_cpf AS
SELECT c.unique_id
FROM {SPLINK_INPUT_VIEW} c
WHERE c.origem = 'censo'
  AND NOT EXISTS (
    SELECT 1 FROM atribuicao_2 a
    WHERE a.unique_id_censo = c.unique_id
  )
''')

con.execute(f'''
CREATE OR REPLACE TABLE melhor_faixa AS
SELECT
    p.unique_id_censo,
    p.unique_id_cpf,
    p.match_probability,
    p.match_weight
FROM splink_predictions p
JOIN censo_sem_cpf s ON s.unique_id = p.unique_id_censo
WHERE p.match_probability >= {PISO}
  AND p.match_probability < {TETO}
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY p.unique_id_censo
    ORDER BY p.match_probability DESC, p.unique_id_cpf
) = 1
''')

display(con.execute(f'''
SELECT
    n_sem_cpf,
    n_com_faixa,
    n_sem_cpf - n_com_faixa AS n_sem_p005,
    ROUND(100.0 * n_com_faixa / NULLIF(n_sem_cpf, 0), 1) AS pct_com_faixa,
    ROUND(100.0 * (n_sem_cpf - n_com_faixa) / NULLIF(n_sem_cpf, 0), 1)
        AS pct_sem_p005
FROM (
    SELECT
        (SELECT COUNT(*) FROM censo_sem_cpf) AS n_sem_cpf,
        (SELECT COUNT(*) FROM melhor_faixa) AS n_com_faixa
)
''').df())


,n_sem_cpf,n_com_faixa,n_sem_p005,pct_com_faixa,pct_sem_p005
0,1986642,216421,1770221,10.9,89.1


Níveis na ordem do treino: nome (Exact → Prefixo → DL≤1 → tokens → JW → Else)
e data (Null → Exact → troca → dia → mês → ano±1 → DL≤1 → DL≤2 → Else).


In [4]:
split_c = "string_split(ca.nome_completo_phon, ' ')"
split_p = "string_split(pb.nome_completo_phon, ' ')"
n_min = f'least(len({split_c}), len({split_p}))'
prefixo_sql = (
    f'{n_min} >= 2 AND len({split_c}) <> len({split_p}) AND '
    f'list_slice({split_c}, 1, {n_min}) = list_slice({split_p}, 1, {n_min})'
)
jw_ultimo_095_sql = (
    f"{split_c}[-1] = {split_p}[-1] AND "
    'jaro_winkler_similarity(ca.nome_completo_phon, pb.nome_completo_phon) >= 0.95'
)
jw_ultimo_092_sql = (
    f"{split_c}[-1] = {split_p}[-1] AND "
    'jaro_winkler_similarity(ca.nome_completo_phon, pb.nome_completo_phon) >= 0.92'
)
dl_completo_1_sql = (
    'damerau_levenshtein(ca.nome_completo_phon, pb.nome_completo_phon) <= 1'
)
n_tok = f'len({split_c})'
mesmo_n = f'{n_tok} = len({split_p}) AND {n_tok} >= 3'
zip_tok = f'list_zip({split_c}, {split_p})'
dif_count = (
    'list_aggregate(list_transform('
    f'{zip_tok}, '
    "x -> CASE WHEN x[1] <> x[2] THEN 1 ELSE 0 END"
    "), 'sum')"
)
dif_max_dl = (
    'list_aggregate(list_transform('
    f'{zip_tok}, '
    "x -> CASE WHEN x[1] <> x[2] "
    "THEN damerau_levenshtein(x[1], x[2]) ELSE 0 END"
    "), 'max')"
)
um_token_sql = (
    f'{mesmo_n} AND {split_c}[-1] = {split_p}[-1] '
    f'AND {dif_count} = 1 AND {dif_max_dl} <= 2'
)
dois_tokens_sql = (
    f'{mesmo_n} AND {dif_count} BETWEEN 1 AND 2 AND {dif_max_dl} <= 2'
)
n_c = f'len({split_c})'
n_p = f'len({split_p})'
ordem_sql = (
    f'{n_c} = {n_p} AND {n_c} >= 3 '
    f'AND list_sort({split_c}) = list_sort({split_p}) '
    f'AND {split_c} <> {split_p}'
)

ors_token = []
for k in range(1, 11):
    if k == 1:
        sem_c = f'list_slice({split_c}, 2, {n_c})'
        sem_p = f'list_slice({split_p}, 2, {n_p})'
    else:
        sem_c = (
            f'list_concat(list_slice({split_c}, 1, {k - 1}), '
            f'list_slice({split_c}, {k + 1}, {n_c}))'
        )
        sem_p = (
            f'list_concat(list_slice({split_p}, 1, {k - 1}), '
            f'list_slice({split_p}, {k + 1}, {n_p}))'
        )
    ors_token.append(f'({n_c} >= {k} AND {sem_c} = {split_p})')
    ors_token.append(f'({n_p} >= {k} AND {sem_p} = {split_c})')
token_a_mais_sql = (
    f'abs({n_c} - {n_p}) = 1 AND least({n_c}, {n_p}) >= 3 AND ('
    + ' OR '.join(ors_token)
    + ')'
)

null_data_sql = (
    '(ca.data_nascimento IS NULL OR pb.data_nascimento IS NULL) AND '
    '(ca.mes_nascimento IS NULL OR pb.mes_nascimento IS NULL OR '
    'ca.dia_nascimento IS NULL OR pb.dia_nascimento IS NULL)'
)
troca_mes_dia_sql = (
    'ca.ano_nascimento = pb.ano_nascimento AND '
    'ca.mes_nascimento = pb.dia_nascimento AND '
    'ca.dia_nascimento = pb.mes_nascimento AND '
    'ca.mes_nascimento <> ca.dia_nascimento'
)
dia_diferente_sql = (
    'ca.ano_nascimento = pb.ano_nascimento AND '
    'ca.mes_nascimento = pb.mes_nascimento AND '
    'ca.dia_nascimento <> pb.dia_nascimento'
)
mes_diferente_sql = (
    'ca.ano_nascimento = pb.ano_nascimento AND '
    'ca.dia_nascimento = pb.dia_nascimento AND '
    'ca.mes_nascimento <> pb.mes_nascimento'
)
ano_pm1_sql = (
    'ca.mes_nascimento = pb.mes_nascimento AND '
    'ca.dia_nascimento = pb.dia_nascimento AND '
    'abs(try_cast(ca.ano_nascimento AS INTEGER) - '
    'try_cast(pb.ano_nascimento AS INTEGER)) = 1'
)

con.execute(f'''
CREATE OR REPLACE TABLE faixa_niveis AS
SELECT
    m.unique_id_censo,
    m.unique_id_cpf,
    m.match_probability,
    m.match_weight,
    ca.nome_completo_phon AS nome_censo,
    pb.nome_completo_phon AS nome_cpf,
    ca.data_nascimento AS dob_censo,
    pb.data_nascimento AS dob_cpf,
    ca.idade AS idade_censo,
    pb.idade AS idade_cpf,
    ca.cep AS cep_censo,
    pb.cep AS cep_cpf,
    ca.logradouro_norm AS logradouro_censo,
    pb.logradouro_norm AS logradouro_cpf,
    ca.nome_mae_phon AS mae_censo,
    pb.nome_mae_phon AS mae_cpf,
    CASE
        WHEN ca.nome_completo_phon IS NULL OR pb.nome_completo_phon IS NULL
            THEN 'null'
        WHEN ca.nome_completo_phon = pb.nome_completo_phon
            THEN 'exato'
        WHEN {prefixo_sql}
            THEN 'prefixo'
        WHEN {dl_completo_1_sql}
            THEN 'dl_1'
        WHEN {um_token_sql}
            THEN 'um_token_dl2'
        WHEN {dois_tokens_sql}
            THEN 'dois_tokens_dl2'
        WHEN {token_a_mais_sql}
            THEN 'token_a_mais'
        WHEN {ordem_sql}
            THEN 'ordem'
        WHEN {jw_ultimo_095_sql}
            THEN 'jw_095'
        WHEN {jw_ultimo_092_sql}
            THEN 'jw_092'
        ELSE 'else'
    END AS nivel_nome,
    CASE
        WHEN {null_data_sql}
            THEN 'null'
        WHEN ca.data_nascimento = pb.data_nascimento
            THEN 'exato'
        WHEN {troca_mes_dia_sql}
            THEN 'mes_dia_trocados'
        WHEN {dia_diferente_sql}
            THEN 'dia_diferente'
        WHEN {mes_diferente_sql}
            THEN 'mes_diferente'
        WHEN {ano_pm1_sql}
            THEN 'ano_pm1'
        WHEN damerau_levenshtein(ca.data_nascimento, pb.data_nascimento) <= 1
            THEN 'dl_1'
        WHEN damerau_levenshtein(ca.data_nascimento, pb.data_nascimento) <= 2
            THEN 'dl_2'
        ELSE 'else'
    END AS nivel_data
FROM melhor_faixa m
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = m.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = m.unique_id_cpf
''')
print('Pares classificados:', con.execute('SELECT COUNT(*) FROM faixa_niveis').fetchone()[0])


Pares classificados: 216421


In [5]:
display(con.execute('''
SELECT
    nivel_nome,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct,
    ROUND(AVG(match_probability), 3) AS avg_p,
    ROUND(AVG(match_weight), 2) AS avg_w
FROM faixa_niveis
GROUP BY 1
ORDER BY n DESC
''').df())


,nivel_nome,n,pct,avg_p,avg_w
0,exato,174394,80.6,0.110,-3.21
1,dl_1,22413,10.4,0.144,-2.72
2,um_token_dl2,19023,8.8,0.077,-3.62
3,ordem,591,0.3,0.064,-3.89


In [6]:
display(con.execute('''
SELECT
    nivel_data,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct,
    ROUND(AVG(match_probability), 3) AS avg_p,
    ROUND(AVG(match_weight), 2) AS avg_w
FROM faixa_niveis
GROUP BY 1
ORDER BY n DESC
''').df())


,nivel_data,n,pct,avg_p,avg_w
0,null,80498,37.2,0.128,-2.91
1,else,69720,32.2,0.063,-3.93
2,dl_2,37147,17.2,0.120,-3.05
3,dia_diferente,13386,6.2,0.183,-2.17
4,dl_1,8356,3.9,0.125,-2.90
5,mes_diferente,4269,2.0,0.213,-1.91
6,ano_pm1,2778,1.3,0.107,-3.22
7,mes_dia_trocados,267,0.1,0.118,-3.02


In [7]:
display(con.execute('''
SELECT
    nivel_nome,
    nivel_data,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct,
    ROUND(AVG(match_probability), 3) AS avg_p
FROM faixa_niveis
GROUP BY 1, 2
ORDER BY n DESC
''').df())


,nivel_nome,nivel_data,n,pct,avg_p
0,exato,else,69720,32.2,0.063
1,exato,null,69267,32.0,0.134
2,exato,dl_2,22601,10.4,0.154
3,um_token_dl2,dl_2,14546,6.7,0.068
4,dl_1,null,11231,5.2,0.090
5,dl_1,dia_diferente,8602,4.0,0.188
6,exato,dl_1,4903,2.3,0.132
7,exato,dia_diferente,4784,2.2,0.174
8,um_token_dl2,dl_1,3227,1.5,0.116
9,dl_1,mes_diferente,2580,1.2,0.228


Amostra dos cruzamentos mais frequentes (nome ELSE, data null, exato+null, etc.).


In [8]:
display(con.execute('''
SELECT
    match_probability,
    nivel_nome,
    nivel_data,
    nome_censo,
    nome_cpf,
    dob_censo,
    dob_cpf,
    idade_censo,
    idade_cpf,
    cep_censo,
    cep_cpf,
    logradouro_censo,
    logradouro_cpf,
    mae_censo,
    mae_cpf
FROM faixa_niveis
WHERE nivel_nome = 'else'
ORDER BY match_probability DESC
LIMIT 40
''').df())

display(con.execute('''
SELECT
    match_probability,
    nivel_nome,
    nivel_data,
    nome_censo,
    nome_cpf,
    dob_censo,
    dob_cpf,
    idade_censo,
    idade_cpf,
    cep_censo,
    cep_cpf,
    logradouro_censo,
    logradouro_cpf,
    mae_censo,
    mae_cpf
FROM faixa_niveis
WHERE nivel_nome = 'exato' AND nivel_data = 'null'
ORDER BY match_probability DESC
LIMIT 40
''').df())


,match_probability,nivel_nome,nivel_data,nome_censo,nome_cpf,dob_censo,dob_cpf,idade_censo,idade_cpf,cep_censo,cep_cpf,logradouro_censo,logradouro_cpf,mae_censo,mae_cpf


,match_probability,nivel_nome,nivel_data,nome_censo,nome_cpf,dob_censo,dob_cpf,idade_censo,idade_cpf,cep_censo,cep_cpf,logradouro_censo,logradouro_cpf,mae_censo,mae_cpf
0,0.247441,exato,null,ANTONIU KARLUS PINHEIRU,ANTONIU KARLUS PINHEIRU,None,1969-06-15,53,53,65077635,65000000,DOS HOLANDESES,RAU 50,None,VITALINA ARAUJO PINHEIRO
1,0.247441,exato,null,RAIMUNDU NONATU AUVIS NASIMENTU,RAIMUNDU NONATU AUVIS NASIMENTU,None,1947-08-27,74,74,65390000,65610000,DA IGREJA,DO MATADOURO,None,FRANCISCA CHAGAS ALVES
2,0.247441,exato,null,RAIMUNDU AZEVEDU,RAIMUNDU AZEVEDU,None,1946-02-20,76,76,65269000,65208000,ARIRIS,POV SAO ROMAO,None,EDUARDA AZEVEDO
3,0.247441,exato,null,JOZI DOMINGUS SOARIS,JOZI DOMINGUS SOARIS,None,1938-06-20,84,84,65235000,65235000,02 DE MAIO,None,None,MARIA JUSTINA SOARES
4,0.247441,exato,null,RAIMUNDU AZEVEDU,RAIMUNDU AZEVEDU,None,1946-08-17,75,75,65400000,65080000,DOUTOR ANTONIO JOAQUIM ARAUJO,SAO JOSE,None,JOANA AZEVEDO
5,0.247441,exato,null,FRANSISKU SOUZA KRUS,FRANSISKU SOUZA KRUS,None,1977-12-23,44,44,65390000,65063300,D0 MEIO,C,None,CARLINDA SOUSA CRUZ
6,0.247441,exato,null,FRANSISKU SOUZA KRUS,FRANSISKU SOUZA KRUS,None,1977-12-23,44,44,65635446,65063300,TRES,C,None,CARLINDA SOUSA CRUZ
7,0.247441,exato,null,ANA VITORIA KOSTA SANTUS,ANA VITORIA KOSTA SANTUS,None,2011-11-04,10,10,None,65707000,B,CEARA,None,MARCIA SILVA COSTA
8,0.247441,exato,null,MARIA GUIRAKAS SIUVA SIUVA,MARIA GUIRAKAS SIUVA SIUVA,None,1962-02-28,60,60,65709000,65715000,PRINCIPAL,LIBERCIA GONCALVES DE MORAIS,None,ADILINA MARIA SILVA
9,0.247441,exato,null,MARIA JOZI DINIS,MARIA JOZI DINIS,None,1960-10-18,61,61,65138000,65120201,JESUS,JOAO DOMASIO PINHEIRO,None,ELZA DINIZ


In [9]:
con.close()